In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

df = pd.read_csv("/content/Batting (1).csv")
df.columns = [c.strip() for c in df.columns]

# Clean numeric columns
num_cols = ["R", "B", "M", "4s", "6s", "SR", "Chase", "Win"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["Date"] = df["Date"].astype(str).str.strip()

# Normalize known inconsistent team-name spellings/casing found in the raw data
TEAM_FIX = {
    "new zealand": "New Zealand",
    "zimbabwae": "Zimbabwe",
    "zimbabwe": "Zimbabwe",
    "eng": "England",
    "england": "England",
    "afganisthan": "Afghanistan",
    "netherland": "Netherlands",
    "sri lanka": "Sri Lanka",
}
def norm_team(x):
    key = str(x).strip().lower()
    return TEAM_FIX.get(key, str(x).strip())

df["Team"] = df["Team"].apply(norm_team)
df["Vs"] = df["Vs"].apply(norm_team)

df["MatchID"] = df["Date"] + " | " + df["Team"] + " vs " + df["Vs"]
# Canonical fixture key: same match regardless of which team's row we're looking at
df["FixtureKey"] = df.apply(
    lambda r: r["Date"] + " | " + " vs ".join(sorted([r["Team"], r["Vs"]])), axis=1
)

print("Rows:", len(df))
print(df.dtypes)
print(df.head())

results = {}

# ---------------------------------------------------------------
# Q1: Top 10 batsmen by total runs across all matches
# ---------------------------------------------------------------
q1 = (df.groupby("Player", as_index=False)["R"].sum()
        .sort_values("R", ascending=False).head(10))
q1.columns = ["Player", "Total Runs"]
results["Q1"] = q1
print("\n=== Q1: Top 10 run scorers ===")
print(q1.to_string(index=False))

# ---------------------------------------------------------------
# Q2: Top 10 highest strike rate, min 50 balls faced (aggregate across matches)
# ---------------------------------------------------------------
agg = df.groupby("Player", as_index=False).agg(Runs=("R", "sum"), Balls=("B", "sum"))
agg = agg[agg["Balls"] >= 50].copy()
agg["StrikeRate"] = (agg["Runs"] / agg["Balls"] * 100).round(2)
q2 = agg.sort_values("StrikeRate", ascending=False).head(10)[["Player", "Balls", "Runs", "StrikeRate"]]
results["Q2"] = q2
print("\n=== Q2: Top 10 Strike Rate (>=50 balls faced) ===")
print(q2.to_string(index=False))

# ---------------------------------------------------------------
# Q3: Most fours, most sixes (career totals)
# ---------------------------------------------------------------
fours_tot = df.groupby("Player")["4s"].sum().sort_values(ascending=False)
sixes_tot = df.groupby("Player")["6s"].sum().sort_values(ascending=False)
most_fours_player, most_fours_val = fours_tot.index[0], fours_tot.iloc[0]
most_sixes_player, most_sixes_val = sixes_tot.index[0], sixes_tot.iloc[0]
q3 = pd.DataFrame({
    "Category": ["Most Fours", "Most Sixes"],
    "Player": [most_fours_player, most_sixes_player],
    "Total": [most_fours_val, most_sixes_val]
})
results["Q3"] = q3
results["Q3_fours_top10"] = fours_tot.head(10).reset_index().rename(columns={"4s": "Total Fours"})
results["Q3_sixes_top10"] = sixes_tot.head(10).reset_index().rename(columns={"6s": "Total Sixes"})
print("\n=== Q3: Most fours & sixes ===")
print(q3.to_string(index=False))

# ---------------------------------------------------------------
# Q4: Which team has won the highest number of matches
# ---------------------------------------------------------------
match_level = df.drop_duplicates(subset=["MatchID", "Team"])[["MatchID", "Team", "Win"]]
team_wins = match_level.groupby("Team")["Win"].sum().sort_values(ascending=False)
top_team = team_wins.index[0]
top_team_wins = int(team_wins.iloc[0])
q4 = team_wins.reset_index().rename(columns={"Win": "Matches Won"})
results["Q4"] = q4
print("\n=== Q4: Team wins ===")
print(q4.to_string(index=False))
print(f"Top team: {top_team} with {top_team_wins} wins")

# ---------------------------------------------------------------
# Q5: For each match, highest run scorer -> Player, Team, Opponent, Date, Runs
# Grouped by FixtureKey so both teams' innings in the same match are compared.
# ---------------------------------------------------------------
idx = df.groupby("FixtureKey")["R"].idxmax()
q5 = df.loc[idx, ["Player", "Team", "Vs", "Date", "R"]].sort_values("Date")
q5.columns = ["Player", "Team", "Opponent", "Match Date", "Runs"]
results["Q5"] = q5
print(f"\nTrue fixture count: {df['FixtureKey'].nunique()}")
print("\n=== Q5: Highest run scorer per match (sample) ===")
print(q5.head(15).to_string(index=False))

# ---------------------------------------------------------------
# Q6: Chasing vs batting first performance
# ---------------------------------------------------------------
chase_grp = df.groupby("Chase").agg(
    AvgRuns=("R", "mean"),
    AvgSR=("SR", "mean"),
    Matches=("MatchID", "nunique"),
    Wins=("Win", "sum")
).round(2)
chase_grp.index = chase_grp.index.map({0: "Batting First", 1: "Chasing"})
chase_grp.index.name = "Situation"
q6 = chase_grp.reset_index()
results["Q6"] = q6
print("\n=== Q6: Chasing vs Batting First ===")
print(q6.to_string(index=False))

# ---------------------------------------------------------------
# Q7: Top bowlers - NOT AVAILABLE (no bowling dataset provided)
# ---------------------------------------------------------------
print("\n=== Q7: Top bowlers by wickets/economy ===")
print("NOT AVAILABLE - dataset contains only batting data (no bowling figures).")

# ---------------------------------------------------------------
# Q8: Runs through boundaries vs strike rate
# ---------------------------------------------------------------
b = df.groupby("Player", as_index=False).agg(Runs=("R", "sum"), Fours=("4s", "sum"),
                                              Sixes=("6s", "sum"), Balls=("B", "sum"))
b["BoundaryRuns"] = b["Fours"] * 4 + b["Sixes"] * 6
b["BoundaryPct"] = (b["BoundaryRuns"] / b["Runs"] * 100).round(2)
b["StrikeRate"] = (b["Runs"] / b["Balls"] * 100).round(2)
q8 = b.sort_values("BoundaryRuns", ascending=False).head(10)[
    ["Player", "Runs", "BoundaryRuns", "BoundaryPct", "StrikeRate"]]
results["Q8"] = q8
print("\n=== Q8: Top 10 by boundary runs, with strike rate ===")
print(q8.to_string(index=False))

# ---------------------------------------------------------------
# Q9: Player Performance Score (runs, SR, 4s, 6s, balls faced) -> Top 20 batsmen
# Weighted formula (explained):
#   Score = Runs*1.0 + SR*0.5 + 4s*1.0 + 6s*2.0 + BallsFaced*0.1
# Rationale: Runs are the primary contributor; strike rate rewards scoring speed
# (half-weighted since it's already correlated with runs/balls); boundaries are
# weighted by their run value (six worth more than a four); balls faced gets a
# small weight to reward durability/consistency without dominating the score.
# ---------------------------------------------------------------
p = df.groupby("Player", as_index=False).agg(
    Runs=("R", "sum"), Balls=("B", "sum"), Fours=("4s", "sum"), Sixes=("6s", "sum")
)
p["SR"] = (p["Runs"] / p["Balls"] * 100).round(2)
p["PerformanceScore"] = (p["Runs"] * 1.0 + p["SR"] * 0.5 + p["Fours"] * 1.0 +
                          p["Sixes"] * 2.0 + p["Balls"] * 0.1).round(2)
q9 = p.sort_values("PerformanceScore", ascending=False).head(20)[
    ["Player", "Runs", "SR", "Fours", "Sixes", "Balls", "PerformanceScore"]]
results["Q9"] = q9
print("\n=== Q9: Top 20 Player Performance Score ===")
print(q9.to_string(index=False))

# ---------------------------------------------------------------
# Q10: All-round performers - NOT FULLY AVAILABLE (no bowling data: wickets/economy)
# Provide a batting-only "Match Impact Score" adaptation instead, using
# runs, strike rate, boundaries and match result (Win), clearly noting the
# missing bowling component.
# Score = Runs*1.0 + SR*0.3 + BoundaryRuns*0.5 + Win*20
# ---------------------------------------------------------------
imp = df.groupby("Player", as_index=False).agg(
    Runs=("R", "sum"), Balls=("B", "sum"), Fours=("4s", "sum"), Sixes=("6s", "sum"),
    Wins=("Win", "sum"), Innings=("MatchID", "nunique")
)
imp["SR"] = (imp["Runs"] / imp["Balls"] * 100).round(2)
imp["BoundaryRuns"] = imp["Fours"] * 4 + imp["Sixes"] * 6
imp["MatchImpactScore"] = (imp["Runs"] * 1.0 + imp["SR"] * 0.3 +
                            imp["BoundaryRuns"] * 0.5 + imp["Wins"] * 20).round(2)
q10 = imp.sort_values("MatchImpactScore", ascending=False).head(10)[
    ["Player", "Runs", "SR", "BoundaryRuns", "Wins", "MatchImpactScore"]]
results["Q10"] = q10
print("\n=== Q10: Top 10 (batting-only Match Impact Score; bowling data unavailable) ===")
print(q10.to_string(index=False))

# Save all tables to CSVs for chart-building step
import os
os.makedirs("tables", exist_ok=True)
for k, v in results.items():
    v.to_csv(f"tables/{k}.csv", index=False)

print("\nSaved tables to ./tables/")

Rows: 745
Date           object
Team           object
Vs             object
Player         object
Dismissal      object
R               int64
B               int64
M               int64
4s              int64
6s              int64
SR            float64
Chase           int64
Win             int64
MatchID        object
FixtureKey     object
dtype: object
     Date         Team        Vs              Player         Dismissal   R   B   M  4s  6s      SR  Chase  Win  \
0  7.2.26  Netherlands  Pakistan      Michael Levitt  c Afridi b Nawaz  24  15  23   3   1  160.00      0    0   
1  7.2.26  Netherlands  Pakistan          Max O'Dowd  c ?Usman b Mirza   5   7  16   1   0   71.42      0    0   
2  7.2.26  Netherlands  Pakistan        Bas de Leede   c Babar b Nawaz  30  25  42   4   0  120.00      0    0   
3  7.2.26  Netherlands  Pakistan     Colin Ackermann     b Abrar Ahmed  20  14  16   4   0  142.85      0    0   
4  7.2.26  Netherlands  Pakistan  Scott Edwards (c)?  c Farhan b Abrar  37  